# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuratulainAzhar22/flyrank-ml-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis:** One row represents one content item for one client on one reporting date.

**Time window:** I use March 2026 (`month = '2026-03'`) as the mid-panel development and verification window. The observed dates span from March 1, 2026 through March 31, 2026.

**Decision moment:** The decision moment is the end of the observed reporting day. Features must use information available on or before that reporting date.

**Prediction/ranking target:** I use future organic-search performance as the outcome proxy, specifically future GSC clicks for the same client and content item.

**Deliberate exclusion:** Future-period performance data is excluded from the feature set because it is unavailable at the decision moment and would create target leakage.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("My_Read_Token")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection configured.")

Hugging Face connection configured.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

I use five historical performance features:

1. **`gsc_impressions`** — observed Google Search Console impressions.
2. **`gsc_clicks`** — observed Google Search Console clicks.
3. **`gsc_avg_position`** — observed average search position.
4. **`ga4_pageviews`** — observed GA4 pageviews.
5. **`sessions_organic`** — observed organic sessions.

These fields describe performance that has already been observed at the decision moment. The warehouse schema contains these GSC and analytics fields.

### Label

**Future GSC clicks** are used as the future-performance proxy. The label represents clicks observed after the March decision window.

### Context

* `client_hash_id` — hashed client identifier.
* `content_hash_id` — hashed content identifier.
* `report_date` — reporting date.
* `month` — monthly partition.
* `client_has_gsc` / `client_has_ga4` — source availability context.
* `gsc_data_available` / `ga4_data_available` — observation-level data availability context.

### Excluded

Future GSC performance fields are excluded from the feature set. In particular, future clicks must not be used as a feature because future clicks are the outcome being predicted. Including them would leak the answer into the inputs.

Client names, URLs, and other identifying information are also excluded because the analysis only requires the hashed identifiers.




## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 — Verify the grain /  Grain

Because the claimed grain is:

client + content + report_date

we need to prove that there aren't duplicate rows at that grain.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' ||
                         content_hash_id || '|' ||
                         CAST(report_date AS VARCHAR)) AS distinct_grain_rows,
        COUNT(*) - COUNT(DISTINCT client_hash_id || '|' ||
                                   content_hash_id || '|' ||
                                   CAST(report_date AS VARCHAR)) AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    """
)

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────┐
│ total_rows │ distinct_grain_rows │ duplicate_rows │
│   int64    │        int64        │     int64      │
├────────────┼─────────────────────┼────────────────┤
│    9841378 │             9841378 │              0 │
└────────────┴─────────────────────┴────────────────┘

**Observed result:** March 2026 contains 9,841,378 rows and 9,841,378 distinct `client_hash_id + content_hash_id + report_date` combinations. The measured duplicate count is 0, supporting the claimed daily client-content observation grain for this March slice.


Query 2 — March row count + date span

In [6]:
q2 = con.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    """
)

q2

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘

**Observed result:** The March 2026 slice contains 9,841,378 rows and spans March 1, 2026 through March 31, 2026. This confirms that the selected verification window covers the full calendar month.


Query 3 — Availability using IS TRUE

In [7]:
q3 = con.sql(
    f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS FALSE
        ) AS gsc_unavailable_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    """
)

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬──────────────────────┐
│ march_rows │ gsc_available_rows │ gsc_unavailable_rows │
│   int64    │       int64        │        int64         │
├────────────┼────────────────────┼──────────────────────┤
│    9841378 │            3611061 │              6230317 │
└────────────┴────────────────────┴──────────────────────┘

**Observed result:** Of the 9,841,378 March rows, 3,611,061 have `gsc_data_available IS TRUE`, while 6,230,317 do not. Therefore, only the GSC-available subset is suitable for features and analysis that depend on GSC measurements.

That is approximately 36.7% available and 63.3% unavailable, which is a significant limitation.

In [8]:
con.sql(
    f"""
    SELECT COUNT(*) AS rows_surviving_gsc_availability
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    """
)

┌─────────────────────────────────┐
│ rows_surviving_gsc_availability │
│              int64              │
├─────────────────────────────────┤
│                         3611061 │
└─────────────────────────────────┘

**Five-feature frame**

Now we need to actually build the feature frame, not merely name the features.

gsc_impressions

gsc_clicks

gsc_avg_position

ga4_pageviews

sessions_organic

In [9]:
features_march = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        sessions_organic

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    ORDER BY report_date, client_hash_id, content_hash_id
    LIMIT 20
    """
)

features_march

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬────────────────────┬───────────────┬──────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ ga4_pageviews │ sessions_organic │
│         varchar         │         varchar          │    date     │      int64      │   int64    │       double       │     int64     │      int64       │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼────────────────────┼───────────────┼──────────────────┤
│ client_0797ff3a1fc9a6a5 │ content_04c67f3541177192 │ 2026-03-01  │               6 │          0 │  6.166666666666667 │          NULL │             NULL │
│ client_0797ff3a1fc9a6a5 │ content_05acc92c165f4386 │ 2026-03-01  │               7 │          0 │  8.714285714285714 │          NULL │             NULL │
│ client_0797ff3a1fc9a6a5 │ content_0f30e04e709c7b5d │ 2026-03-0

### Five features — available when?

1. **`gsc_impressions`** — Available at the decision moment because it comes from the observed GSC reporting period and does not require future performance data.

2. **`gsc_clicks`** — Available at the decision moment because it records clicks observed during the current reporting period, before the future outcome window.

3. **`gsc_avg_position`** — Available at the decision moment because it summarizes the page's observed search position during the current reporting period.

4. **`ga4_pageviews`** — Available at the decision moment because it records pageviews already observed in the current reporting period.

5. **`sessions_organic`** — Available at the decision moment because it represents organic sessions already recorded during the observed period.


**The leakage experiment**

This is the part I don't want you to fake.

We can deliberately create a feature that is literally derived from the future label.

First, let's create the actual future label.

For March observations, the future month is April 2026.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leak_df = con.sql(
    f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS current_clicks
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS future_clicks
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE month = '2026-04'
          AND gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.current_clicks,
        COALESCE(a.future_clicks, 0) AS future_clicks

    FROM march m
    LEFT JOIN april a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    """
)

leak_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬────────────────┬───────────────┐
│     client_hash_id      │     content_hash_id      │ current_clicks │ future_clicks │
│         varchar         │         varchar          │     int128     │    int128     │
├─────────────────────────┼──────────────────────────┼────────────────┼───────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │              2 │             2 │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │              0 │             0 │
│ client_73cda7b4e4f265ea │ content_905aa32a0230694e │              0 │             0 │
│ client_73cda7b4e4f265ea │ content_05434271b257bb68 │              6 │            30 │
│ client_73cda7b4e4f265ea │ content_d056587ff7faca0c │             16 │             6 │
│ client_73cda7b4e4f265ea │ content_bfd1e41c2af250c8 │              0 │             0 │
│ client_73cda7b4e4f265ea │ content_2662845f598544ef │              1 │             0 │
│ client_73cda7b4e4f265ea │ cont

**Deliberately leak the label**

I deliberately added the future outcome, `future_clicks`, as an input feature and trained a quick regression model.

**Leaky R²: 0.998056899572395**

The score jumped to approximately 1.0 because the model was given a variable derived from the outcome it was supposed to predict. This is not legitimate predictive performance; it is target leakage.

I therefore removed `future_clicks` from the feature set. The honest feature set contains only historical/current-period information:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_pageviews`
* `sessions_organic`

The 0.9981 leaky score is retained only as evidence of the leakage trap and is not treated as the model's real performance.


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

df_leak = leak_df.df()

X = df_leak[["future_clicks"]]
y = df_leak["future_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model_leak = DecisionTreeRegressor(random_state=42)

model_leak.fit(X_train, y_train)

pred_leak = model_leak.predict(X_test)

leak_score = r2_score(y_test, pred_leak)

print("Leaky R²:", leak_score)

Leaky R²: 0.998056899572395


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


A major limitation is uneven GSC availability. In the March 2026 slice, 3,611,061 of 9,841,378 rows have `gsc_data_available IS TRUE`, while 6,230,317 rows do not. Therefore, GSC-based analysis applies only to the available subset and may not represent the entire warehouse population.

The data also cannot establish causality. It records observed search and analytics performance but does not capture every external factor that may influence future performance.

History may also be unbalanced across clients and content items, and GSC and GA4 availability can differ. I therefore treat the results as measured and directional decision-support rather than causal evidence.

The June 2026 `_sample` is treated as a sealed final month and is not used to develop the future-performance label.


In [12]:
honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic"
]
label = "future_clicks"

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.